# The Perceptron — the first trainable neuron (1958)

> Tutorial pair for [`perceptron.py`](perceptron.py).

## 1. Intuition
A perceptron draws a straight line and says "this side = +1, that side = −1." It
**learns from its mistakes**: every time it misclassifies a point, it nudges the
line toward getting that point right. This mistake-driven rule is the historical
ancestor of all neural networks.

## 2. Concept (the slide)
- **Model:** $\hat y=\operatorname{sign}(\mathbf w^\top\mathbf x+b)$, labels in $\{-1,+1\}$.
- **Learning rule:** only update on a mistake.
- **Guarantee:** if the data is **linearly separable**, it converges in a finite
  number of updates (Novikoff). If not, it never settles → use the *pocket*
  (keep the best-so-far) or *averaged* variant.
- **Limit:** a single perceptron cannot represent XOR → you need a hidden layer.

## 3. Math derivation

**Perceptron loss.** Define $\ell(\mathbf w)=\max(0,\,-y(\mathbf w^\top\mathbf x+b))$
— zero when correctly classified, otherwise the (negative) margin. Its
(sub)gradient on a misclassified point is $-y\mathbf x$. One step of SGD gives the
classic rule:

$$\boxed{\;\mathbf w\leftarrow\mathbf w+\eta\,y\,\mathbf x,\qquad b\leftarrow b+\eta\,y\;}\quad\text{(only when }y(\mathbf w^\top\mathbf x+b)\le 0).$$

**Why the update helps.** After the update the new margin on that point is
$y(\mathbf w+\eta y\mathbf x)^\top\mathbf x = y\mathbf w^\top\mathbf x+\eta\lVert\mathbf x\rVert^2$,
which is **larger** by $\eta\lVert\mathbf x\rVert^2$ — pushing toward correctness.

**Convergence (Novikoff).** If there is a unit $\mathbf w^\star$ with margin
$\gamma=\min_i y_i\mathbf w^{\star\top}\mathbf x_i>0$ and $\lVert\mathbf x_i\rVert\le R$,
the number of updates is bounded by $(R/\gamma)^2$ — independent of dimension and
dataset size. (Proof tracks $\mathbf w_t^\top\mathbf w^\star$ growing linearly while
$\lVert\mathbf w_t\rVert$ grows at most like $\sqrt t$.)

**XOR.** No single hyperplane separates $\{(0,0),(1,1)\}$ from $\{(0,1),(1,0)\}$;
the best any line does is 3/4. This concrete failure is exactly what a hidden
layer (the MLP) fixes.

## 4. NumPy implementation (vanilla / pocket / averaged / multiclass)

In [ ]:
# ===== actual implementation from perceptron.py =====
from __future__ import annotations

import numpy as np

SEED = 0

class PerceptronNumPy:
    r"""
    Prediction:  ŷ = sign(w·x + b),  labels in {-1, +1}.
    Update on a mistake (y·(w·x+b) ≤ 0):
        w ← w + η y x ,   b ← b + η y
    This is stochastic (sub)gradient descent on the hinge-at-0 loss
        max(0, -y (w·x + b)).
    """

    def __init__(self, lr=1.0, n_epochs=20, mode="vanilla", seed=SEED):
        self.lr, self.n_epochs, self.mode, self.seed = lr, n_epochs, mode, seed

    def fit(self, X, y):
        X = np.asarray(X, float); y = np.where(np.asarray(y) <= 0, -1, 1)
        n, d = X.shape
        rng = np.random.default_rng(self.seed)
        w = np.zeros(d); b = 0.0
        w_sum = np.zeros(d); b_sum = 0.0; count = 0
        best_w, best_b, best_err = w.copy(), b, n + 1
        self.errors_ = []
        for _ in range(self.n_epochs):
            errs = 0
            for i in rng.permutation(n):
                if y[i] * (w @ X[i] + b) <= 0:        # misclassified
                    w += self.lr * y[i] * X[i]
                    b += self.lr * y[i]
                    errs += 1
                w_sum += w; b_sum += b; count += 1     # for averaged
            self.errors_.append(errs)
            if errs < best_err:                        # pocket: remember best
                best_err, best_w, best_b = errs, w.copy(), b
            if errs == 0:
                break
        if self.mode == "pocket":
            self.w, self.b = best_w, best_b
        elif self.mode == "averaged":
            self.w, self.b = w_sum / count, b_sum / count
        else:
            self.w, self.b = w, b
        return self

    def decision_function(self, X):
        return np.asarray(X, float) @ self.w + self.b

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1)

class MulticlassPerceptron:
    """One-vs-rest wrapper around the binary perceptron."""

    def __init__(self, n_classes, **kw):
        self.n_classes, self.kw = n_classes, kw

    def fit(self, X, y):
        self.models = []
        for c in range(self.n_classes):
            yc = (np.asarray(y) == c).astype(int)
            self.models.append(PerceptronNumPy(**self.kw).fit(X, yc))
        return self

    def predict(self, X):
        scores = np.stack([m.decision_function(X) for m in self.models], 1)
        return scores.argmax(1)

## 5. PyTorch implementation (single linear unit, perceptron loss)

In [ ]:
# ===== actual implementation from perceptron.py =====
import torch

import torch.nn as nn

class PerceptronTorch(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.lin = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.lin(x).squeeze(-1)

    def fit(self, X, y, lr=0.1, n_epochs=50):
        dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        y = torch.as_tensor(np.where(np.asarray(y) <= 0, -1, 1),
                            dtype=torch.float32, device=dev)
        opt = torch.optim.SGD(self.parameters(), lr=lr)
        for _ in range(n_epochs):
            opt.zero_grad()
            margin = y * self(X)
            loss = torch.clamp(-margin, min=0).mean()   # perceptron loss
            loss.backward(); opt.step()
        return self

    @torch.no_grad()
    def predict(self, X):
        dev = next(self.parameters()).device
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        return torch.where(self(X) >= 0, 1, -1).cpu().numpy()

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    from sklearn.datasets import make_blobs

    X, y = make_blobs(n_samples=200, centers=2, cluster_std=1.0, random_state=SEED)
    p = PerceptronNumPy(n_epochs=20).fit(X, y)
    converged = p.errors_[-1] == 0
    print(f"ran {len(p.errors_)} epochs (zero-error reached: {converged}); "
          f"final mistakes/epoch={p.errors_[-1]}, "
          f"acc={np.mean((p.predict(X) > 0).astype(int) == y):.3f}")

    pt = PerceptronTorch(2).fit(X, y)
    print(f"torch acc={np.mean((pt.predict(X) > 0).astype(int) == y):.3f}")

    # XOR: not linearly separable — the perceptron cannot fit it (motivates MLP)
    Xx = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
    yx = np.array([0, 1, 1, 0])
    px = PerceptronNumPy(n_epochs=100, mode="pocket").fit(Xx, yx)
    print(f"XOR best accuracy a single perceptron reaches: "
          f"{np.mean((px.predict(Xx) > 0).astype(int) == yx):.2f} (max 0.75 — needs a hidden layer)")

## 6. Train — separable data, and the XOR failure

In [ ]:
demo()

## 7. Visualization — the boundary and mistakes per epoch

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
import perceptron as M

X, y = make_blobs(n_samples=200, centers=2, cluster_std=0.8, random_state=0)
p = M.PerceptronNumPy(n_epochs=15).fit(X, y)
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200),
                     np.linspace(X[:,1].min()-1, X[:,1].max()+1, 200))
zz = p.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].contourf(xx, yy, zz, alpha=.3, cmap="bwr")
ax[0].scatter(X[:,0], X[:,1], c=y, edgecolor="k", s=15, cmap="bwr")
ax[0].set_title("Learned linear boundary")
ax[1].plot(p.errors_, "o-"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("mistakes")
ax[1].set_title("Mistakes per epoch")
plt.tight_layout(); plt.show()

## 8. Takeaways
- The perceptron = SGD on the perceptron loss; updates only on mistakes.
- Guaranteed convergence **iff** linearly separable; otherwise use pocket/averaged.
- It outputs a hard label, no probability → logistic regression softens it; SVM
  maximizes the margin instead of just *a* separating line.
- XOR ⇒ we need depth → **[MLP](../../dl/mlp/mlp.ipynb)**.